In [ ]:
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt
import numpy as np
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay

from modules import ConvNeXt
from dataset import create_datasets, create_dataloaders

In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print("Using:", device)

In [ ]:
train_dir = "/content/drive/MyDrive/Colab Notebooks/Pattern Recognition Project/AD_NC/train"
test_dir  = "/content/drive/MyDrive/Colab Notebooks/Pattern Recognition Project/AD_NC/test"

In [ ]:
_, _, test_ds = create_datasets(train_dir, test_dir)
_, _, test_loader = create_dataloaders(_, _, test_ds)

In [ ]:
model = ConvNeXt(in_chans=1, num_classes=2).to(device)
model.load_state_dict(torch.load("best_convnext_adni.pth", map_location=device))
model.eval()

In [ ]:
all_preds, all_targets = [], []
with torch.no_grad():
    for data, targets in test_loader:
        data, targets = data.to(device), targets.to(device)
        outputs = model(data)
        probs = F.softmax(outputs, dim=1)
        preds = probs.argmax(1)
        all_preds.extend(preds.cpu().numpy())
        all_targets.extend(targets.cpu().numpy())

In [ ]:
print("\n📋 Classification Report:")
print(classification_report(all_targets, all_preds, digits=4))

In [ ]:
cm = confusion_matrix(all_targets, all_preds)
disp = ConfusionMatrixDisplay(cm, display_labels=["AD", "NC"])
disp.plot(cmap='Blues'); plt.show()